In [11]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
from torchvision import transforms as trn

import clip
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from tqdm import tqdm

from data_loading import load_sun_data
from dataloaders import create_sun_dataloader
from cav_utils import (calculate_filtered_cav, calculate_centroid_cav,
                       get_cosine_similarity, get_dot_product_similarity, 
                       get_concept_distribution_stats, is_obs_in_concept)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Używane urządzenie: {DEVICE}")

Używane urządzenie: cuda


In [2]:
model_file = 'models/resnet18_places365.pth'
conv_model = models.resnet18(num_classes=365)
checkpoint = torch.load(model_file, map_location=lambda storage, loc: storage)
state_dict = {str.replace(k, 'module.', ''): v for k, v in checkpoint['state_dict'].items()}
conv_model.load_state_dict(state_dict)
conv_model.fc = nn.Identity() 
conv_model = conv_model.to(DEVICE).eval()

centre_crop = trn.Compose([
    trn.Resize((256, 256)),
    trn.CenterCrop(224),
    trn.ToTensor(),
    trn.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

clip_model, clip_preprocess = clip.load('ViT-B/32', device=DEVICE)
clip_model.eval()
print(f'Modele załadowane. Wymiar embeddingu CLIP: {clip_model.visual.output_dim}')

Modele załadowane. Wymiar embeddingu CLIP: 512


In [3]:
df_sun, attributes_list, attr_cols = load_sun_data(base_path='data/SUN/SUNAttributeDB', threshold=0.5)

print(f'Images read: {len(df_sun)}')
print(f'Number of attributes: {len(attr_cols)}')

df_sun.head(3)

Images read: 14340
Number of attributes: 102


,image_id,file_path,class_id,sailing/ boating,driving,biking,transporting things or people,sunbathing,vacationing/ touring,hiking,...,far-away horizon,no horizon,rugged scene,mostly vertical components,mostly horizontal components,symmetrical,cluttered space,scary,soothing,stressful
0,1,a/abbey/sun_aakbdcgfpksytcwj.jpg,a,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
1,2,a/abbey/sun_aaoktempcmudsvna.jpg,a,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3,a/abbey/sun_abegcweqnetpdlrh.jpg,a,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [4]:
dataloader_resnet = create_sun_dataloader(
    df=df_sun, img_dir='data/SUN/images/', transform=centre_crop,
    attr_cols=attr_cols, batch_size=32, num_workers=4
)

dataloader_clip = create_sun_dataloader(
    df=df_sun, img_dir='data/SUN/images/', transform=clip_preprocess,
    attr_cols=attr_cols, batch_size=64, num_workers=4)

In [5]:
def extract_clip_features(dataloader, model):
    """Przepuszcza obrazy przez CLIP i zwraca znormalizowane wektory."""
    all_features = []
    all_labels = []
    all_paths = []
    
    #model.eval()
    with torch.no_grad():
        for imgs, labels, paths in tqdm(dataloader, desc="Ekstrakcja cech CLIP"):
            features = model.encode_image(imgs.to(DEVICE))
            features = features / features.norm(dim=-1, keepdim=True)
            
            all_features.append(features.cpu().numpy())
            all_labels.append(labels.numpy())
            all_paths.extend(paths)
            
    return np.vstack(all_features), np.vstack(all_labels), all_paths


features_clip, labels_clip, paths_clip = extract_clip_features(dataloader_clip, clip_model)
df_labels = pd.DataFrame(labels_clip, columns=attr_cols)

Ekstrakcja cech CLIP:   0%|          | 0/225 [00:00<?, ?it/s]/home/JanTar/miniconda3/envs/automl/lib/python3.11/site-packages/torch/nn/modules/conv.py:456: UserWarning: Applied workaround for CuDNN issue, install nvrtc.so (Triggered internally at ../aten/src/ATen/native/cudnn/Conv_v8.cpp:84.)
  return F.conv2d(input, weight, bias, self.stride,
Ekstrakcja cech CLIP: 100%|██████████| 225/225 [02:39<00:00,  1.42it/s]


In [6]:
def extract_resnet_features(dataloader, model):
    all_features = []
    
    with torch.no_grad():
        for imgs, labels, paths in tqdm(dataloader, desc="Downloading ResNet features"):
            features = model(imgs.to(DEVICE))
            features = features / features.norm(dim=-1, keepdim=True)
            all_features.append(features.cpu().numpy())
            
    return np.vstack(all_features)

print("Downloading ResNet features...")
features_resnet = extract_resnet_features(dataloader_resnet, conv_model)

print(f"Shape of CLIP features: {features_clip.shape}")
print(f"Shape of ResNet features: {features_resnet.shape}")

Shape of CLIP features: (14340, 512)
Shape of ResNet features: (14340, 512)


In [7]:
indices = np.arange(len(df_sun))
train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=42)

df_train = df_labels.iloc[train_idx].reset_index(drop=True)
df_test  = df_labels.iloc[test_idx].reset_index(drop=True)

X_train_clip = features_clip[train_idx]
X_test_clip  = features_clip[test_idx]

X_train_resnet = features_resnet[train_idx]
X_test_resnet  = features_resnet[test_idx]

print(f"Training set: {len(train_idx)} samples")
print(f"Test set: {len(test_idx)} samples")

def calculate_standard_cav(features_matrix, pos_indices, neg_indices):
    mean_concept = np.mean(features_matrix[pos_indices], axis=0)
    mean_background = np.mean(features_matrix[neg_indices], axis=0)
    
    cav = mean_concept - mean_background
    norm = np.linalg.norm(cav)
    if norm > 1e-8:
        return cav / norm
    raise ValueError('Norma CAV jest bliska zeru.')

Training set: 11472 samples
Test set: 2868 samples


In [8]:
hierarchies_to_test = [
    # empirical
    ('research', 'enclosed area'),
    ('camping', 'open area'),
    ('medical activity', 'enclosed area'),
    ('gaming', 'enclosed area'),
    ('sand', 'natural light'),
    ('carpet', 'enclosed area'),

    # sematical
    ('ocean', 'open area'),
    ('conducting business', 'working'),
    ('pavement', 'asphalt'),
    ('climbing', 'rock/stone'), 
    ('rusty', 'aged/ worn'),  
    ('ice', 'cold')
]

def evaluate_cav_methods(X_train, X_test, df_train, df_test, pairs, model_name):
    print("\n" + "="*80)
    print(f"EWALUATION FOR MODEL: {model_name.upper()}")
    print("="*80)
    print(f"{'SUBCONCEPT':<18} | {'PARENT':<15} | {'AUC STD CAV':<15} | {'AUC FILTERED CAV':<16}")
    print("-" * 80)
    
    results = []
    
    for child, parent in pairs:
        # --- TRENING ---
        # 1. Indeksy dla standardowego CAV (cały zbiór treningowy)
        child_pos_train = df_train.index[df_train[child] == 1.0].to_numpy()
        child_neg_train = df_train.index[df_train[child] == 0.0].to_numpy()
        
        # 2. Indeksy dla Filtered CAV (pojęcie rodzica musi istnieć)
        parent_pos_train = df_train.index[df_train[parent] == 1.0].to_numpy()
        
        # Zabezpieczenie przed brakiem danych
        if len(child_pos_train) == 0 or len(parent_pos_train) == 0:
            continue
            
        try:
            # Trenujemy oba wektory
            cav_std = calculate_standard_cav(X_train, child_pos_train, child_neg_train)
            cav_filtered = calculate_filtered_cav(X_train, child_pos_train, child_neg_train, parent_pos_train)
        except ValueError:
            continue # Pomijamy jeśli brakuje danych w przecięciu (Filtered)
            
            
        # --- TESTOWANIE (TWARDE NEGATYWY) ---
        # Filtrujemy zbiór testowy do obrazów, gdzie WYSTĘPUJE pojęcie rodzica
        hard_mask_test = df_test[parent] == 1.0
        
        y_true_hard = df_test[child][hard_mask_test].values
        X_hard = X_test[hard_mask_test]
        
        # Sprawdzamy czy mamy pozytywy i negatywy do wyliczenia AUC
        if len(np.unique(y_true_hard)) > 1:
            # Przewidywania (podobieństwo kosinusowe)
            scores_std = cosine_similarity(X_hard, cav_std.reshape(1, -1)).flatten()
            scores_filtered = cosine_similarity(X_hard, cav_filtered.reshape(1, -1)).flatten()
            
            auc_std = roc_auc_score(y_true_hard, scores_std)
            auc_filtered = roc_auc_score(y_true_hard, scores_filtered)
            diff = auc_filtered - auc_std


            results.append({
                'child': child, 'parent': parent, 
                'auc_std': auc_std, 
                'auc_filtered': auc_filtered,
                'auc_diff': diff
            })
            
            znak = "+++" if auc_filtered > auc_std else "---"
            print(f"{child:<18} | {parent:<15} | {auc_std:<15.4f} | {auc_filtered:<15.4f} {znak} | Diff: {diff:+.4f}")
        else:
            print(f"{child:<18} | {parent:<15} | {'---':<15} | {'--- (brak podziału w teście)'}")

    return pd.DataFrame(results)

# Uruchamiamy eksperyment
df_results_clip = evaluate_cav_methods(
    X_train_clip, X_test_clip, df_train, df_test, hierarchies_to_test, "CLIP (ViT-B/32)"
)

df_results_resnet = evaluate_cav_methods(
    X_train_resnet, X_test_resnet, df_train, df_test, hierarchies_to_test, "ResNet18 (Places365)"
)


EWALUATION FOR MODEL: CLIP (VIT-B/32)
SUBCONCEPT         | PARENT          | AUC STD CAV     | AUC FILTERED CAV
--------------------------------------------------------------------------------
research           | enclosed area   | 0.9186          | 0.9136          --- | Diff: -0.0050
camping            | open area       | 0.8798          | 0.8935          +++ | Diff: +0.0137
medical activity   | enclosed area   | 0.9739          | 0.9701          --- | Diff: -0.0039
gaming             | enclosed area   | 0.9506          | 0.9589          +++ | Diff: +0.0083
sand               | natural light   | 0.8858          | 0.9015          +++ | Diff: +0.0158
carpet             | enclosed area   | 0.7604          | 0.7628          +++ | Diff: +0.0024
ocean              | open area       | 0.9600          | 0.9627          +++ | Diff: +0.0027
conducting business | working         | 0.7745          | 0.6962          --- | Diff: -0.0783
pavement           | asphalt         | 0.6000          | 0.59

### ASD

In [9]:
indices = np.arange(len(df_sun))
train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=42)

df_train = df_labels.iloc[train_idx].reset_index(drop=True)
df_test  = df_labels.iloc[test_idx].reset_index(drop=True)

# Podział cech CLIP
X_train_clip = features_clip[train_idx]
X_test_clip  = features_clip[test_idx]

# Podział cech ResNet
X_train_resnet = features_resnet[train_idx]
X_test_resnet  = features_resnet[test_idx]

print(f"Zbiór treningowy: {len(train_idx)} próbek")
print(f"Zbiór testowy: {len(test_idx)} próbek")

all_pairs = [
    # empirical
    ('research', 'enclosed area'),
    ('camping', 'open area'),
    ('medical activity', 'enclosed area'),
    ('gaming', 'enclosed area'),
    ('sand', 'natural light'),
    ('carpet', 'enclosed area'),

    # sematical
    ('ocean', 'open area'),
    ('conducting business', 'working'),
    ('pavement', 'asphalt'),
    ('climbing', 'rock/stone'), 
    ('rusty', 'aged/ worn'),  
    ('ice', 'cold')
]

child_to_parent = {a: b for a, b in all_pairs}

Zbiór treningowy: 11472 próbek
Zbiór testowy: 2868 próbek


In [13]:

def is_obs_in_concept(X_obs, cav_parent, mu_parent, sigma_parent, 
                      cav_child, mu_child, sigma_child, prob_z_thresh=0.95):
    """
    Cascade inference for single new observation X_obs.
    Returns a bool tuple (exists_parent_B, exists_child_A)


    similarity_score >= (mu - z_threshold * sigma)
    Checks if the similarity score falls 
    within the concept distribution using one sided Z-Score: (x - mu)/ sigma >= -z_threshold
    """
    if sigma_parent == 0:
        return False, False

    X_obs = np.asarray(X_obs).flatten()
    cav_parent = np.asarray(cav_parent).flatten()
    cav_child = np.asarray(cav_child).flatten()
    
    norm_obs = np.linalg.norm(X_obs)
    norm_parent = np.linalg.norm(cav_parent)
    norm_child = np.linalg.norm(cav_child)

    if norm_obs == 0 or norm_parent == 0 or norm_child == 0: 
        return False, False

    Z_SCORES = {
    0.75: 0.67449, 
    0.80: 0.84162,
    0.85: 1.03643,
    0.90: 1.28155,
    0.95: 1.64485,
    0.975: 1.95996,
    0.99: 2.32634,
    0.999: 3.09024
    }

    z_thresh = Z_SCORES.get(prob_z_thresh, 1.64485) # default to 1.64485 for 95% if not found
    
    sim_b = np.dot(X_obs, cav_parent) / (norm_obs * norm_parent)
    #sim_b = cosine_similarity(X_obs, cav_parent.reshape(1, -1)).flatten()[0]
    exists_b = bool( sim_b >= (mu_parent - z_thresh * sigma_parent) )

    if not exists_b:
        return False, False
    if sigma_child == 0:
        return True, False
    
    # sim_a = cosine_similarity(X_obs, cav_child.reshape(1, -1)).flatten()[0]
    sim_a = np.dot(X_obs, cav_child) / (norm_obs * norm_child)
    exists_a = bool( sim_a >= (mu_child - z_thresh * sigma_child) )
    
    return True, exists_a

In [17]:
def evaluate_cascade_inference(X_train, X_test, df_train, df_test, pairs, model_name, prob_z=0.95):
    print("\n" + "="*110)
    print(f" EWALUACJA INFERENCJI KASKADOWEJ: {model_name.upper()} (prob={prob_z})")
    print("="*110)
    print(f"{'SUBCONCEPT':<20} | {'PARENT':<18} | {'AUC STD CASCADE':<16} | {'AUC FILT CASCADE':<16} | {'DIFF'}")
    print("-" * 110)
    
    valid_results = []
    invalid_results = []
    
    for child, parent in pairs:
        # --- 1. TRENING (Wektory i Statystyki) ---
        y_train_c = df_train[child].values
        y_train_p = df_train[parent].values
        
        idx_c_pos = np.where(y_train_c == 1.0)[0]
        idx_c_neg = np.where(y_train_c == 0.0)[0]
        idx_p_pos = np.where(y_train_p == 1.0)[0]
        idx_p_neg = np.where(y_train_p == 0.0)[0]
        
        if len(idx_c_pos) < 2 or len(idx_p_pos) < 2:
            invalid_results.append({'child': child, 'parent': parent, 'reason': 'Za mało pozytywów w treningu'})
            continue
            
        try:
            # Trenujemy wektor Rodzica
            cav_parent = calculate_centroid_cav(X_train, idx_p_pos, idx_p_neg)
            
            # Trenujemy warianty Dziecka
            cav_child_std = calculate_centroid_cav(X_train, idx_c_pos, idx_c_neg)
            cav_child_filt = calculate_filtered_cav(X_train, idx_c_pos, idx_c_neg, idx_p_pos)
            
        except ValueError:
            invalid_results.append({'child': child, 'parent': parent, 'reason': 'Błąd obliczania CAV'})
            continue
            
        # Wyciągamy statystyki rozkładu ze zbiorów TRENINGOWYCH
        try:
            mu_p, sigma_p = get_concept_distribution_stats(X_train, y_train_p, cav_parent)
            mu_c_std, sigma_c_std = get_concept_distribution_stats(X_train, y_train_c, cav_child_std)
            mu_c_filt, sigma_c_filt = get_concept_distribution_stats(X_train, y_train_c, cav_child_filt)
        except ValueError:
             invalid_results.append({'child': child, 'parent': parent, 'reason': 'Błąd statystyk (za mało próbek)'})
             continue


        # --- 2. INFERENCJA (Zbiór testowy) ---
        y_true_test = df_test[child].values
        
        # Inicjalizujemy wyniki testowe (domyślnie rzucamy karę za brak przejścia kaskady)
        KARA_PUNKTOWA = -1.0 
        scores_cascade_std = np.full(len(X_test), KARA_PUNKTOWA)
        scores_cascade_filt = np.full(len(X_test), KARA_PUNKTOWA)
        
        # Testujemy każdą próbkę (symulacja online)
        for i in range(len(X_test)):
            obs = X_test[i]
            
            # Wnioskowanie w wariancie Standard (Kaskada 1)
            has_p_std, has_c_std = is_obs_in_concept(
                obs, cav_parent, mu_p, sigma_p, 
                cav_child_std, mu_c_std, sigma_c_std, prob_z_thresh=prob_z
            )
            
            if has_p_std:
                # Obraz ma rodzica - przypisz mu prawdziwy wynik podobieństwa do dziecka
                # (is_obs_in_concept wylicza ten score w środku, ale dla AUC musimy go znać. 
                # Zamiast go wyciągać ze środka, najbezpieczniej policzyć go tu)
                sim_std = np.dot(obs, cav_child_std) / (np.linalg.norm(obs) * np.linalg.norm(cav_child_std))
                scores_cascade_std[i] = sim_std
                
                
            # Wnioskowanie w wariancie Filtered (Kaskada 2)
            has_p_filt, has_c_filt = is_obs_in_concept(
                obs, cav_parent, mu_p, sigma_p, 
                cav_child_filt, mu_c_filt, sigma_c_filt, prob_z_thresh=prob_z
            )
            
            if has_p_filt:
                # Obraz ma rodzica - przypisz mu zaktualizowany wynik
                sim_filt = np.dot(obs, cav_child_filt) / (np.linalg.norm(obs) * np.linalg.norm(cav_child_filt))
                scores_cascade_filt[i] = sim_filt

                
        # --- 3. EWALUACJA ---
        if len(np.unique(y_true_test)) > 1:
            auc_std = roc_auc_score(y_true_test, scores_cascade_std)
            auc_filt = roc_auc_score(y_true_test, scores_cascade_filt)
            diff = auc_filt - auc_std
            
            valid_results.append({
                'child': child, 'parent': parent, 
                'auc_std': auc_std, 'auc_filt': auc_filt, 'diff': diff
            })
        else:
             invalid_results.append({'child': child, 'parent': parent, 'reason': 'Brak etykiet testowych'})


    # --- SORTOWANIE I WYDRUK ---
    valid_results.sort(key=lambda x: x['diff'], reverse=True)
    
    for res in valid_results:
        znak = "+++" if res['diff'] > 0 else "---" if res['diff'] < 0 else "==="
        print(f"{res['child']:<20} | {res['parent']:<18} | {res['auc_std']:<16.4f} | {res['auc_filt']:<16.4f} | {znak} Diff: {res['diff']:+.4f}")
        
    for res in invalid_results:
        print(f"{res['child']:<20} | {res['parent']:<18} | {'---':<16} | {'---':<16} | {res['reason']}")

    return pd.DataFrame(valid_results)

# ==========================================
# Uruchomienie na obu modelach
# ==========================================

print("Eksperyment z łagodnym progiem Z-Score (prob=0.90):")
df_cascade_clip = evaluate_cascade_inference(X_train_clip, X_test_clip, df_train, df_test, hierarchies_to_test, "CLIP (ViT-B/32)", prob_z=0.85)

df_cascade_resnet = evaluate_cascade_inference(X_train_resnet, X_test_resnet, df_train, df_test, hierarchies_to_test, "ResNet18 (Places)", prob_z=0.85)

Eksperyment z łagodnym progiem Z-Score (prob=0.90):

 EWALUACJA INFERENCJI KASKADOWEJ: CLIP (VIT-B/32) (prob=0.85)
SUBCONCEPT           | PARENT             | AUC STD CASCADE  | AUC FILT CASCADE | DIFF
--------------------------------------------------------------------------------------------------------------
sand                 | natural light      | 0.9153           | 0.9202           | +++ Diff: +0.0049
camping              | open area          | 0.9321           | 0.9362           | +++ Diff: +0.0041
gaming               | enclosed area      | 0.9527           | 0.9561           | +++ Diff: +0.0034
carpet               | enclosed area      | 0.9077           | 0.9097           | +++ Diff: +0.0020
rusty                | aged/ worn         | 0.7100           | 0.7112           | +++ Diff: +0.0012
ocean                | open area          | 0.9517           | 0.9526           | +++ Diff: +0.0009
research             | enclosed area      | 0.9706           | 0.9700           | --- D